# AdaptaBrasil API — exploration

This notebook loads **indicator metadata** from the hierarchy endpoint and pulls **map values** for a chosen indicator using the public API documented by [AdaptaBrasilAPIAccess](https://github.com/AdaptaBrasil/AdaptaBrasilAPIAccess).

- **Portal:** [adaptabrasil.mcti.gov.br](https://adaptabrasil.mcti.gov.br/)
- **API host:** `https://sistema.adaptabrasil.mcti.gov.br`

In [66]:
from __future__ import annotations

import json
from typing import Any

import pandas as pd
import requests

BASE_URL = "https://sistema.adaptabrasil.mcti.gov.br"
SCHEMA = "adaptabrasil"
DEFAULT_RECORTE = "BR"
DEFAULT_RESOLUCAO = "municipio"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CityCatalyst-dataset-review/1.0"})

## 1. Indicator hierarchy (metadata only)

Same URL used by the official Python script: `/api/hierarquia/{schema}`.

In [67]:
def fetch_hierarchy(base_url: str = BASE_URL, schema: str = SCHEMA, timeout: int = 60) -> list[dict[str, Any]]:
    url = f"{base_url.rstrip('/')}/api/hierarquia/{schema}"
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


indicators_raw = fetch_hierarchy()
len(indicators_raw)

558

In [68]:
def parse_years_cell(y: Any) -> list[int]:
    if y is None or (isinstance(y, float) and pd.isna(y)):
        return []
    if isinstance(y, list):
        out: list[int] = []
        for x in y:
            try:
                out.append(int(x))
            except (TypeError, ValueError):
                continue
        return out
    s = str(y).strip()
    if not s:
        return []
    return [int(p) for p in (x.strip() for x in s.split(",")) if p.isdigit()]


def indicators_to_frame(rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.json_normalize(rows)
    if "years" in df.columns:
        df["years_list"] = df["years"].apply(parse_years_cell)
    return df


indicators = indicators_to_frame(indicators_raw)
# Drop API level-1 sector nodes (no municipal maps in the same way); they stay in `indicators` for sector names
with_data = indicators[indicators["level"].astype(int) > 1].copy()
with_data[["id", "name", "level", "measurement_unit", "years_list"]].head(10)

,id,name,level,measurement_unit,years_list
10,2,Risco de estresse hídrico,2,None,"[2020, 2030, 2050]"
11,5001,Disponibilidade de alimentos,2,None,"[2017, 2030, 2050]"
12,5048,Acesso e consumo de alimentos,2,None,"[2017, 2030, 2050]"
13,10001,Acesso,2,None,"[2019, 2055]"
14,10024,Disponibilidade,2,None,"[2019, 2055]"
15,40001,Tempestade,2,None,"[2020, 2030, 2050]"
16,40029,Vendaval,2,None,"[2020, 2030, 2050]"
17,40056,Aumento do nível do mar,2,None,"[2020, 2030, 2050]"
18,50001,Malária,2,None,"[2020, 2030, 2050]"
19,50030,Leishmaniose tegumentar americana,2,None,"[2020, 2030, 2050]"


In [69]:
with_data.to_csv('./sample/adapta_indicator_metadata.csv', index=False)


In [64]:
# Hierarquia: uma caminhada na API (indicator_id_master) + colunas derivadas
_full_by_id = {int(r["id"]): r for _, r in indicators.iterrows()}


def _path_chain(iid: int) -> list[int]:
    """IDs do setor (nível 1) até a folha."""
    parts: list[int] = []
    cur = int(iid)
    for _ in range(40):
        parts.append(cur)
        row = _full_by_id.get(cur)
        if row is None:
            break
        m = row.get("indicator_id_master")
        if m is None or (isinstance(m, float) and pd.isna(m)):
            break
        try:
            m_int = int(m)
        except (TypeError, ValueError):
            break
        if m_int == 0:
            break
        cur = m_int
    return list(reversed(parts))


def _cell(cid: int, key: str) -> str:
    r = _full_by_id.get(cid)
    if r is None:
        return ""
    v = r.get(key)
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return str(int(v)) if key == "level" else str(v)


def _names_join(chain: list[int], start: int) -> str:
    if start >= len(chain):
        return ""
    names = [_cell(chain[i], "name") for i in range(start, len(chain))]
    return " › ".join(n for n in names if n)


_chains = with_data["id"].astype(int).map(_path_chain)
with_data["sector_root_id"] = _chains.map(lambda c: c[0] if c else pd.NA).astype("Int64")
with_data["sector_root_name"] = _chains.map(lambda c: _cell(c[0], "name") if c else "")
with_data["path_full"] = _chains.map(lambda c: _names_join(c, 0))
with_data["path_no_root"] = _chains.map(lambda c: _names_join(c, 1))
with_data["root_name"] = _chains.map(lambda c: _cell(c[1], "name") if len(c) >= 2 else "")
with_data["previous_level_name"] = _chains.map(
    lambda c: _cell(c[-2], "name") if len(c) >= 2 else ""
)

_MAX = 6
for _k in range(1, _MAX + 1):
    with_data[f"id_level_{_k}"] = _chains.map(lambda c, k=_k: str(c[k - 1]) if len(c) >= k else "")
    with_data[f"level_level_{_k}"] = _chains.map(
        lambda c, k=_k: _cell(c[k - 1], "level") if len(c) >= k else ""
    )
    with_data[f"name_level_{_k}"] = _chains.map(
        lambda c, k=_k: _cell(c[k - 1], "name") if len(c) >= k else ""
    )
with_data["path_depth"] = _chains.map(len)

_preview = [
    "id",
    "name",
    "level",
    "path_depth",
    "id_level_1",
    "level_level_1",
    "name_level_1",
    "id_level_2",
    "level_level_2",
    "name_level_2",
    "id_level_3",
    "level_level_3",
    "name_level_3",
    "path_full",
]
with_data.sort_values(
    ["sector_root_id", "path_depth", "id"], ascending=[True, False, True]
).loc[:, _preview].head(12)


,id,name,level,path_depth,id_level_1,level_level_1,name_level_1,id_level_2,level_level_2,name_level_2,id_level_3,level_level_3,name_level_3,path_full
284,19,Ineficiência na produção da água,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
285,20,Ineficiência na distribuição da água,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
286,21,Consumo médio per capita de água,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
287,22,Balanço hídrico para agropecuária,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
288,23,Balanço hídrico para indústria,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
289,24,Doenças devido ao saneamento inadequado,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
290,25,Qualidade da água,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
291,26,Áreas degradadas e/ou desmatadas,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
292,27,Áreas com solos susceptíveis à erosão,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...
293,28,Vazão ecológica para usos ecossistêmicos,6,6,1,1,Recursos hídricos,2,2,Risco de estresse hídrico,3,3,Vulnerabilidade,Recursos hídricos › Risco de estresse hídrico ...


In [ ]:
_hierarchy_csv_cols = [
    "id",
    "name",
    "level",
    "path_depth",
    "sector_root_id",
    "sector_root_name",
    "root_name",
    "previous_level_name",
    "path_no_root",
    "path_full",
] + [x for k in range(1, 7) for x in (f"id_level_{k}", f"level_level_{k}", f"name_level_{k}")]

_sorted_h = with_data.sort_values(
    ["sector_root_id", "path_depth", "id"], ascending=[True, False, True]
)
_sorted_h.loc[:, _hierarchy_csv_cols].to_csv(
    "./sample/adapta_indicator_hierarchy_paths.csv", index=False
)

# Estrutura: caminhos maximais — nome_nivel_* e id_nivel_* (IDs do AdaptaBrasil por nível)
_nivel_cols = [f"name_level_{k}" for k in range(1, 7)]
_id_cols = [f"id_level_{k}" for k in range(1, 7)]
_seen: set[tuple[str, ...]] = set()
_name_prefix_to_ids: dict[tuple[str, ...], tuple[str, ...]] = {}


def _norm_name_cell(v: object) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return str(v).strip()


def _norm_id_cell(v: object) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    if s == "":
        return ""
    try:
        return str(int(float(s)))
    except (TypeError, ValueError):
        return s


for _tup_n, _tup_i in zip(
    _sorted_h[_nivel_cols].itertuples(index=False, name=None),
    _sorted_h[_id_cols].itertuples(index=False, name=None),
):
    _parts = [_norm_name_cell(v) for v in _tup_n]
    _idparts = [_norm_id_cell(v) for v in _tup_i]
    while _parts and _parts[-1] == "":
        _parts.pop()
    while _idparts and _idparts[-1] == "":
        _idparts.pop()
    if len(_parts) != len(_idparts):
        raise ValueError(f"depth mismatch: names={_parts!r} ids={_idparts!r}")
    for _d in range(1, len(_parts) + 1):
        _seen.add(tuple(_parts[:_d] + [""] * (6 - _d)))
        _ntp = tuple(_parts[:_d])
        _itp = tuple(_idparts[:_d])
        _prev = _name_prefix_to_ids.get(_ntp)
        if _prev is not None and _prev != _itp:
            raise ValueError(f"ambiguous name path {_ntp}: {_prev} vs {_itp}")
        _name_prefix_to_ids[_ntp] = _itp


def _trim_nivel_path(t: tuple[str, ...]) -> tuple[str, ...]:
    lst = list(t)
    while lst and lst[-1] == "":
        lst.pop()
    return tuple(lst)


_trimmed = {_trim_nivel_path(t) for t in _seen}
_maximal = {
    p
    for p in _trimmed
    if not any(len(q) > len(p) and q[: len(p)] == p for q in _trimmed)
}
_tabela_cols = [f"nome_nivel_{k}" for k in range(1, 7)] + [f"id_nivel_{k}" for k in range(1, 7)]
_tabela_rows: list[list[str]] = []
for p in sorted(_maximal):
    _ids = _name_prefix_to_ids[p]
    _tabela_rows.append(
        list(p) + [""] * (6 - len(p)) + list(_ids) + [""] * (6 - len(_ids))
    )
pd.DataFrame(_tabela_rows, columns=_tabela_cols).to_csv(
    "./data/adapta_indicator_hierarchy.csv", index=False
)


## 2. Map data for one indicator

Pattern: `/api/mapa-dados/{recorte}/{resolucao}/{indicator_id}/{year}/{scenario}/{schema}`

Use `null` for no scenario when the indicator has no pessimist/optimist split (passed as the literal string `null` in the path).

In [43]:
def first_available_year(row: pd.Series) -> int | None:
    years = row.get("years_list") or []
    return int(years[0]) if years else None


def build_mapa_dados_url(
    indicator_id: int,
    year: int,
    *,
    recorte: str = DEFAULT_RECORTE,
    resolucao: str = DEFAULT_RESOLUCAO,
    scenario: str = "null",
    schema: str = SCHEMA,
) -> str:
    return (
        f"{BASE_URL}/api/mapa-dados/{recorte}/{resolucao}/{indicator_id}/{year}/{scenario}/{schema}"
    )


def fetch_mapa_dados(url: str, timeout: int = 120) -> Any:
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


# Pick the first indicator that has at least one year
sample_row = with_data[with_data["years_list"].apply(len) > 0].iloc[0]
INDICATOR_ID = int(sample_row["id"])
YEAR = first_available_year(sample_row)
assert YEAR is not None

map_url = build_mapa_dados_url(INDICATOR_ID, YEAR)
print(sample_row["name"], "| id=", INDICATOR_ID, "| year=", YEAR)
print(map_url)

Risco de estresse hídrico | id= 2 | year= 2020
https://sistema.adaptabrasil.mcti.gov.br/api/mapa-dados/BR/municipio/2/2020/null/adaptabrasil


In [44]:
map_payload = fetch_mapa_dados(map_url)
type(map_payload), (list(map_payload.keys()) if isinstance(map_payload, dict) else "list-like")

(list, 'list-like')

In [45]:
def map_payload_to_frame(payload: Any) -> pd.DataFrame:
    """Normalize mapa-dados JSON into a flat table (best-effort — schema may vary)."""
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        if "features" in payload:
            return pd.json_normalize(payload["features"])
        if "data" in payload:
            return map_payload_to_frame(payload["data"])
        return pd.json_normalize([payload])
    raise TypeError(f"Unexpected payload type: {type(payload)}")


def parse_br_decimal(value: Any) -> float | None:
    """API often returns numbers as strings using comma as decimal separator."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip()
    if not s:
        return None
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


map_df = map_payload_to_frame(map_payload)
map_df.head()

,id,geocod_ibge,name,indicator_id,year,scenario_id,pessimist,value,valuecolor,rangelabel
0,2644,5200050,Abadia de Goiás/GO,2,2020,None,1,0.66,#FF8300,Alto
1,754,3100104,Abadia dos Dourados/MG,2,2020,None,1,0.28,#A9DE00,Baixo
2,2645,5200100,Abadiânia/GO,2,2020,None,1,0.35,#A9DE00,Baixo
3,755,3100203,Abaeté/MG,2,2020,None,1,0.29,#A9DE00,Baixo
4,2938,1500107,Abaetetuba/PA,2,2020,None,1,0.48,#FFCD00,Médio


In [46]:
map_df.to_csv('./sample/adapta_indicators.csv', index=False)


In [78]:
# Build modeled hierarchy table from adapta_indicator_hierarchy.csv
hierarchy_in = "./data/adapta_indicator_hierarchy.csv"
modeled_out = "./data/adapta_indicator_hierarchy_modeled.csv"

h = pd.read_csv(hierarchy_in, dtype="string").fillna("")

name_cols = [f"nome_nivel_{n}" for n in [1, 2, 3, 4, 5, 6]]
id_cols = [f"id_nivel_{n}" for n in [1, 2, 3, 4, 5, 6]]

rows = []
for _, r in h.iterrows():
    # Deepest non-empty level defines the base indicator.
    leaf_level = max(
        [n for n in [1, 2, 3, 4, 5, 6] if str(r[f"id_nivel_{n}"]).strip() != ""],
        default=3,
    )

    # Impact chain starts after level 3 and stops before the base indicator level.
    impact_levels = [n for n in [4, 5, 6] if n < leaf_level]

    out = {
        "sector_id": r["id_nivel_1"],
        "sector_name": r["nome_nivel_1"],
        "risk_id": r["id_nivel_2"],
        "risk_name": r["nome_nivel_2"],
        "risk_component_id": r["id_nivel_3"],
        "risk_component_name": r["nome_nivel_3"],
        "base_indicator_id": r[f"id_nivel_{leaf_level}"],
        "base_indicator_name": r[f"nome_nivel_{leaf_level}"],
        "base_indicator_level": str(leaf_level),
    }

    # Keep fixed columns for easy downstream use.
    for i, n in enumerate(impact_levels, start=1):
        out[f"impact_chain_id_{i}"] = r[f"id_nivel_{n}"]
        out[f"impact_chain_name_{i}"] = r[f"nome_nivel_{n}"]
    for i in range(len(impact_levels) + 1, 4):
        out[f"impact_chain_id_{i}"] = ""
        out[f"impact_chain_name_{i}"] = ""

    rows.append(out)

modeled_df = pd.DataFrame(rows)

# Standardize risk component names.
hazard_aliases = {"Ameaça Climática", "Ameaça climática", "Ameaça de escassez hídrica"}
modeled_df.loc[modeled_df["risk_component_name"].isin(hazard_aliases), "risk_component_name"] = "Ameaça"

# These should exist only as base indicators (no risk_component_* value).
base_only_names = {
    "Potencial de energia solar",
    "Potencial de energia hidrelétrica",
    "Potencial de energia eólica",
    "Demanda de resfriamento",
}
mask_base_only = modeled_df["base_indicator_name"].isin(base_only_names) | modeled_df[
    "risk_component_name"
].isin(base_only_names)
modeled_df.loc[mask_base_only, ["risk_component_id", "risk_component_name"]] = ""

modeled_df = modeled_df.drop_duplicates().sort_values(
    ["sector_id", "risk_id", "risk_component_id", "base_indicator_id"]
)

col_order = [
    "sector_id",
    "sector_name",
    "risk_id",
    "risk_name",
    "risk_component_id",
    "risk_component_name",
    "impact_chain_id_1",
    "impact_chain_name_1",
    "impact_chain_id_2",
    "impact_chain_name_2",
    "impact_chain_id_3",
    "impact_chain_name_3",
    "base_indicator_id",
    "base_indicator_name",
    "base_indicator_level",
]
modeled_df[col_order].to_csv(modeled_out, index=False)
print(f"Saved: {modeled_out}")

Saved: ./data/adapta_indicator_hierarchy_modeled.csv


In [79]:
# Build English modeled hierarchy table
import json

modeled_in = "./data/adapta_indicator_hierarchy_modeled.csv"
modeled_out_en = "./data/adapta_indicator_hierarchy_modeled_en.csv"
translation_json = "./data/adapta_indicator_pt_to_en.json"

with open(translation_json, "r", encoding="utf-8") as f:
    pt_to_en = json.load(f)

en_modeled = pd.read_csv(modeled_in, dtype="string").fillna("")

name_cols = [
    "sector_name",
    "risk_name",
    "risk_component_name",
    "impact_chain_name_1",
    "impact_chain_name_2",
    "impact_chain_name_3",
    "base_indicator_name",
]

for c in name_cols:
    en_modeled[c] = en_modeled[c].map(lambda x: pt_to_en.get(x, x) if pd.notna(x) else x)

# Standardize risk component names.
hazard_aliases = {"Climate Hazard", "Climate hazard", "Water scarcity hazard"}
en_modeled.loc[
    en_modeled["risk_component_name"].isin(hazard_aliases), "risk_component_name"
] = "Hazard"

# These should exist only as base indicators (no risk_component_* value).
base_only_names = {
    "Solar energy potential",
    "Hydropower potential",
    "Wind energy potential",
    "Cooling Demand",
}
mask_base_only = en_modeled["base_indicator_name"].isin(base_only_names) | en_modeled[
    "risk_component_name"
].isin(base_only_names)
en_modeled.loc[mask_base_only, ["risk_component_id", "risk_component_name"]] = ""

en_modeled.to_csv(modeled_out_en, index=False)
print(f"Saved: {modeled_out_en}")

Saved: ./data/adapta_indicator_hierarchy_modeled_en.csv


In [70]:
# Build city + modeled hierarchy table (PT)
city_csv = "./data/adapta_city_data.csv"
modeled_hierarchy_csv = "./data/adapta_indicator_hierarchy_modeled.csv"
out_csv = "./data/adapta_city_all_levels.csv"

modeled_cols = [
    "sector_id",
    "sector_name",
    "risk_id",
    "risk_name",
    "risk_component_id",
    "risk_component_name",
    "impact_chain_id_1",
    "impact_chain_name_1",
    "impact_chain_id_2",
    "impact_chain_name_2",
    "impact_chain_id_3",
    "impact_chain_name_3",
    "base_indicator_id",
    "base_indicator_name",
    "base_indicator_level",
]


In [72]:
city_df = pd.read_csv(city_csv, dtype={"geocod_ibge": "string"})
modeled_df = pd.read_csv(modeled_hierarchy_csv, dtype="string").fillna("")

# Base city table with defaults.
cities = (
    city_df[["geocod_ibge", "name"]]
    .drop_duplicates()
    .rename(columns={"geocod_ibge": "city_id", "name": "city_name"})
)
cities["timeframe"] = 2020
cities["scenario"] = "current"

# One record per city + modeled hierarchy row.
out_df = cities.merge(modeled_df[modeled_cols], how="cross")

# Indicator lookup used to attach numeric/string values by ID.
value_lookup = (
    city_df[["geocod_ibge", "indicator_id", "value", "rangelabel"]]
    .drop_duplicates(subset=["geocod_ibge", "indicator_id"], keep="last")
    .rename(columns={"geocod_ibge": "city_id"})
)
value_lookup["city_id"] = value_lookup["city_id"].astype("string")
value_lookup["indicator_id"] = value_lookup["indicator_id"].astype("string")

# Add values for all ID columns except sector_id.
id_to_value_prefix = {
    "risk_id": "risk",
    "risk_component_id": "risk_component",
    "impact_chain_id_1": "impact_chain_1",
    "impact_chain_id_2": "impact_chain_2",
    "impact_chain_id_3": "impact_chain_3",
    "base_indicator_id": "base_indicator",
}

for id_col, prefix in id_to_value_prefix.items():
    out_df[id_col] = out_df[id_col].astype("string")

    value_cols = value_lookup.rename(
        columns={
            "indicator_id": id_col,
            "value": f"{prefix}_value_numeric",
            "rangelabel": f"{prefix}_value_string",
        }
    )[["city_id", id_col, f"{prefix}_value_numeric", f"{prefix}_value_string"]]

    out_df = out_df.merge(value_cols, on=["city_id", id_col], how="left")

final_cols = [
    "city_id",
    "city_name",
    "timeframe",
    "scenario",
    "sector_id",
    "sector_name",
    "risk_id",
    "risk_name",
    "risk_value_numeric",
    "risk_value_string",
    "risk_component_id",
    "risk_component_name",
    "risk_component_value_numeric",
    "risk_component_value_string",
    "impact_chain_id_1",
    "impact_chain_name_1",
    "impact_chain_1_value_numeric",
    "impact_chain_1_value_string",
    "impact_chain_id_2",
    "impact_chain_name_2",
    "impact_chain_2_value_numeric",
    "impact_chain_2_value_string",
    "impact_chain_id_3",
    "impact_chain_name_3",
    "impact_chain_3_value_numeric",
    "impact_chain_3_value_string",
    "base_indicator_id",
    "base_indicator_name",
    "base_indicator_level",
    "base_indicator_value_numeric",
    "base_indicator_value_string",
]

out_df[final_cols].to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: ./data/adapta_city_all_levels.csv


In [96]:
# Build English version from corrected PT->EN translations
import json

translation_json = "./data/adapta_indicator_pt_to_en.json"
src_all_levels_csv = "./data/adapta_city_all_levels.csv"
out_all_levels_en_csv = "./data/adapta_city_all_levels_en.csv"

with open(translation_json, "r", encoding="utf-8") as f:
    pt_to_en = json.load(f)

en_df = pd.read_csv(src_all_levels_csv, dtype="string")

name_cols = [
    "sector_name",
    "risk_name",
    "risk_component_name",
    "impact_chain_name_1",
    "impact_chain_name_2",
    "impact_chain_name_3",
    "base_indicator_name",
]
value_string_cols = [
    "risk_value_string",
    "risk_component_value_string",
    "impact_chain_1_value_string",
    "impact_chain_2_value_string",
    "impact_chain_3_value_string",
    "base_indicator_value_string",
]

# Translate names using the JSON map.
for c in name_cols:
    en_df[c] = en_df[c].map(lambda x: pt_to_en.get(x, x) if pd.notna(x) else x)

# Translate categorical labels.
fallback_labels = {
    "Muito baixo": "Very low",
    "Baixo": "Low",
    "Médio": "Medium",
    "Alto": "High",
    "Muito alto": "Very high",
    "Dado indisponível": "Data unavailable",
}
for c in value_string_cols:
    en_df[c] = en_df[c].map(
        lambda x: pt_to_en.get(x, fallback_labels.get(x, x)) if pd.notna(x) else x
    )

# Keep risk_component normalization aligned with modeled hierarchy files.
hazard_aliases = {"Climate Hazard", "Climate hazard", "Water scarcity hazard"}
en_df.loc[en_df["risk_component_name"].isin(hazard_aliases), "risk_component_name"] = "Hazard"

base_only_names = {
    "Solar energy potential",
    "Hydropower potential",
    "Wind energy potential",
    "Cooling Demand",
}
mask_base_only = en_df["base_indicator_name"].isin(base_only_names) | en_df[
    "risk_component_name"
].isin(base_only_names)
en_df.loc[mask_base_only, ["risk_component_id", "risk_component_name"]] = ""

en_df.to_csv(out_all_levels_en_csv, index=False)
print(f"Saved: {out_all_levels_en_csv}")


Saved: ./data/adapta_city_all_levels_en.csv


In [97]:
# MEED-style wide export from sample-one-city.csv, including scenario_id + timeframe (year)
import json
import pandas as pd

sample_path = "./sample/sample-one-city.csv"
modeled_en_path = "./data/adapta_indicator_hierarchy_modeled_en.csv"
translation_json = "./data/adapta_indicator_pt_to_en.json"
out_path = "./data/adapta_sample_data_one_city_with_scenarios.csv"

modeled_cols = [
    "sector_id",
    "sector_name",
    "risk_id",
    "risk_name",
    "risk_component_id",
    "risk_component_name",
    "impact_chain_id_1",
    "impact_chain_name_1",
    "impact_chain_id_2",
    "impact_chain_name_2",
    "impact_chain_id_3",
    "impact_chain_name_3",
    "base_indicator_id",
    "base_indicator_name",
    "base_indicator_level",
]

def norm_scenario_id(x):
    if pd.isna(x) or x == "":
        return ""
    try:
        f = float(x)
        if f == int(f):
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(x).strip()

raw = pd.read_csv(sample_path, dtype={"geocod_ibge": "string"})
raw["scenario_id"] = raw["scenario_id"].map(norm_scenario_id)
raw["timeframe"] = raw["year"].astype(int)

city_keys = (
    raw[["geocod_ibge", "name", "timeframe", "scenario_id"]]
    .drop_duplicates()
    .rename(columns={"geocod_ibge": "city_id", "name": "city_name"})
)
city_keys["city_id"] = city_keys["city_id"].astype("string")

value_lookup = raw[
    ["geocod_ibge", "timeframe", "scenario_id", "indicator_id", "value", "rangelabel"]
].rename(columns={"geocod_ibge": "city_id"})
value_lookup["city_id"] = value_lookup["city_id"].astype("string")
value_lookup["indicator_id"] = value_lookup["indicator_id"].astype(str)

modeled_df = pd.read_csv(modeled_en_path, dtype="string").fillna("")

out_df = city_keys.merge(modeled_df[modeled_cols], how="cross")

id_to_value_prefix = {
    "risk_id": "risk",
    "risk_component_id": "risk_component",
    "impact_chain_id_1": "impact_chain_1",
    "impact_chain_id_2": "impact_chain_2",
    "impact_chain_id_3": "impact_chain_3",
    "base_indicator_id": "base_indicator",
}

for id_col, prefix in id_to_value_prefix.items():
    out_df[id_col] = out_df[id_col].astype(str)
    value_cols = value_lookup.rename(
        columns={
            "indicator_id": id_col,
            "value": f"{prefix}_value_numeric",
            "rangelabel": f"{prefix}_value_string",
        }
    )[
        ["city_id", "timeframe", "scenario_id", id_col, f"{prefix}_value_numeric", f"{prefix}_value_string"]
    ]
    out_df = out_df.merge(value_cols, on=["city_id", "timeframe", "scenario_id", id_col], how="left")

# Match adapta_sample_data_one_city.csv shape: no internal ids; scenario -> scenario_id
export_cols = [
    "city_name",
    "timeframe",
    "scenario_id",
    "sector_name",
    "risk_name",
    "risk_value_numeric",
    "risk_value_string",
    "risk_component_name",
    "risk_component_value_numeric",
    "risk_component_value_string",
    "impact_chain_name_1",
    "impact_chain_1_value_numeric",
    "impact_chain_1_value_string",
    "impact_chain_name_2",
    "impact_chain_2_value_numeric",
    "impact_chain_2_value_string",
    "base_indicator_name",
    "base_indicator_value_numeric",
    "base_indicator_value_string",
]

export_df = out_df[export_cols].copy()

# Optional: align categorical rangelabel strings with existing EN export (same as EN cell)
with open(translation_json, "r", encoding="utf-8") as f:
    pt_to_en = json.load(f)
value_string_cols = [c for c in export_cols if c.endswith("_value_string")]
fallback_labels = {
    "Muito baixo": "Very low",
    "Baixo": "Low",
    "Médio": "Medium",
    "Alto": "High",
    "Muito alto": "Very high",
    "Dado indisponível": "Data unavailable",
}
for c in value_string_cols:
    export_df[c] = export_df[c].map(
        lambda x: pt_to_en.get(x, fallback_labels.get(x, x)) if pd.notna(x) and x != "" else x
    )

export_df.to_csv(out_path, index=False)
print(f"Saved: {out_path} rows={len(export_df)}")

Saved: ./data/adapta_sample_data_one_city_with_scenarios.csv rows=15710
